<a href="https://colab.research.google.com/github/neha-sharma4/ML-2_Lab/blob/main/ML_Lab_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# SEQUENTIAL COVERING ALGORITHM
# ============================================================
# Learn a set of IF-THEN rules from training data
#
# Process:
# 1. Learn one rule using general-to-specific search
# 2. Remove positive examples covered by that rule
# 3. Repeat until no positive examples remain
# ============================================================

import pandas as pd


# ============================================================
# 1. DATASET
# ============================================================

attributes = [
    "Sky",
    "AirTemp",
    "Humidity",
    "Wind",
    "Water",
    "Forecast"
]

data = [
    ["Sunny", "Warm", "Normal", "Strong", "Warm", "Same",   "Yes"],
    ["Sunny", "Warm", "High",   "Strong", "Warm", "Same",   "Yes"],
    ["Rainy", "Cold", "High",   "Strong", "Warm", "Change", "No"],
    ["Sunny", "Warm", "High",   "Strong", "Cool", "Change", "Yes"],
    ["Cloudy", "Warm", "Normal", "Weak", "Warm", "Same",   "Yes"],
    ["Rainy", "Cold", "Normal", "Weak", "Cool", "Change", "No"],
    ["Sunny", "Cold", "Normal", "Weak", "Warm", "Same",   "No"],
    ["Cloudy", "Warm", "High", "Strong", "Cool", "Change", "Yes"]
]

columns = attributes + ["Class"]

df = pd.DataFrame(data, columns=columns)

print("DATASET")
print("=" * 80)
print(df.to_string(index=False))


# ============================================================
# 2. SEPARATE FEATURES AND TARGET
# ============================================================

X = [row[:-1] for row in data]
y = [row[-1] for row in data]


# ============================================================
# 3. HELPER FUNCTIONS
# ============================================================

def covers(rule, instance):
    """
    Checks whether a rule covers an instance.

    '?' means any value is accepted.
    """

    for rule_value, instance_value in zip(rule, instance):

        if rule_value == "?":
            continue

        if rule_value != instance_value:
            return False

    return True


def accuracy(rule, X, y):
    """
    Calculates the accuracy of a rule.

    A good rule should:
    - cover many positive examples
    - cover few negative examples
    """

    correct = 0

    for instance, label in zip(X, y):

        prediction = covers(rule, instance)

        actual = (label == "Yes")

        if prediction == actual:
            correct += 1

    return correct / len(X)


def positives_covered(rule, X, y):
    """
    Returns the number of positive examples
    covered by the rule.
    """

    count = 0

    for instance, label in zip(X, y):

        if label == "Yes" and covers(rule, instance):
            count += 1

    return count


def negatives_covered(rule, X, y):
    """
    Returns the number of negative examples
    covered by the rule.
    """

    count = 0

    for instance, label in zip(X, y):

        if label == "No" and covers(rule, instance):
            count += 1

    return count


# ============================================================
# 4. GENERATE SPECIALIZATIONS
# ============================================================

def generate_specializations(rule, domains):
    """
    Generate all immediate specializations of a rule.

    Example:

    (?, ?, ?, ?, ?, ?)

    can become:

    (Sunny, ?, ?, ?, ?, ?)
    (Rainy, ?, ?, ?, ?, ?)
    (Cloudy, ?, ?, ?, ?, ?)

    etc.
    """

    specializations = []

    for i in range(len(rule)):

        # We can specialize only a generalized attribute
        if rule[i] == "?":

            for value in domains[i]:

                new_rule = list(rule)

                new_rule[i] = value

                specializations.append(
                    tuple(new_rule)
                )

    return specializations


# ============================================================
# 5. LEARN-ONE-RULE
# ============================================================

def learn_one_rule(X, y, attributes, beam_width=5):

    # --------------------------------------------------------
    # Find possible values of every attribute
    # --------------------------------------------------------

    domains = []

    for i in range(len(attributes)):

        values = set()

        for instance in X:
            values.add(instance[i])

        domains.append(sorted(values))


    # --------------------------------------------------------
    # Start with the most general rule
    # --------------------------------------------------------

    rule = tuple(["?"] * len(attributes))

    beam = [rule]


    print("\n")
    print("=" * 80)
    print("LEARN-ONE-RULE")
    print("=" * 80)

    print("\nInitial rule:")
    print(rule)


    # --------------------------------------------------------
    # General-to-specific beam search
    # --------------------------------------------------------

    while True:

        # If current rule covers no negative examples,
        # it is a pure rule
        if negatives_covered(rule, X, y) == 0:
            break


        # Generate specializations
        candidates = []

        for r in beam:

            new_rules = generate_specializations(
                r,
                domains
            )

            candidates.extend(new_rules)


        # Remove duplicate rules
        candidates = list(set(candidates))


        # ----------------------------------------------------
        # Score candidates
        # ----------------------------------------------------

        scored_candidates = []

        for candidate in candidates:

            pos = positives_covered(
                candidate,
                X,
                y
            )

            neg = negatives_covered(
                candidate,
                X,
                y
            )

            # Rule quality:
            # prioritize positive coverage
            # and penalize negative coverage
            score = pos - neg

            scored_candidates.append(
                (
                    score,
                    pos,
                    -neg,
                    candidate
                )
            )


        # If no candidates exist
        if len(scored_candidates) == 0:
            break


        # Sort from best to worst
        scored_candidates.sort(
            reverse=True
        )


        # Keep only the best beam_width candidates
        beam = [
            item[3]
            for item in scored_candidates[:beam_width]
        ]


        # Select best candidate
        rule = beam[0]


        print("\nCurrent best rule:")
        print(rule)

        print(
            "Positive examples covered:",
            positives_covered(rule, X, y)
        )

        print(
            "Negative examples covered:",
            negatives_covered(rule, X, y)
        )


        # Stop if rule covers no negative examples
        if negatives_covered(rule, X, y) == 0:
            break


    return rule


# ============================================================
# 6. SEQUENTIAL COVERING
# ============================================================

def sequential_covering(X, y, attributes):

    # --------------------------------------------------------
    # Store all learned rules
    # --------------------------------------------------------

    rules = []


    # --------------------------------------------------------
    # Work with copies of the data
    # --------------------------------------------------------

    remaining_X = list(X)
    remaining_y = list(y)


    # --------------------------------------------------------
    # Continue while positive examples exist
    # --------------------------------------------------------

    while "Yes" in remaining_y:

        print("\n\n")
        print("#" * 80)
        print("SEQUENTIAL COVERING")
        print("#" * 80)


        # ----------------------------------------------------
        # Learn one rule
        # ----------------------------------------------------

        rule = learn_one_rule(
            remaining_X,
            remaining_y,
            attributes
        )


        # ----------------------------------------------------
        # Check whether rule actually covers positives
        # ----------------------------------------------------

        covered_positives = []

        for instance, label in zip(
            remaining_X,
            remaining_y
        ):

            if label == "Yes" and covers(
                rule,
                instance
            ):
                covered_positives.append(
                    instance
                )


        # Prevent infinite loop
        if len(covered_positives) == 0:

            print("\nNo additional positive examples can be covered.")
            break


        # ----------------------------------------------------
        # Add rule to final rule set
        # ----------------------------------------------------

        rules.append(rule)


        print("\nLearned Rule:")
        print(rule)

        print(
            "Positive examples covered:",
            len(covered_positives)
        )


        # ----------------------------------------------------
        # Remove all positive examples covered by this rule
        # ----------------------------------------------------

        new_X = []
        new_y = []


        for instance, label in zip(
            remaining_X,
            remaining_y
        ):

            # Remove only covered positive examples
            if label == "Yes" and covers(
                rule,
                instance
            ):
                continue

            new_X.append(instance)
            new_y.append(label)


        remaining_X = new_X
        remaining_y = new_y


        print(
            "Remaining positive examples:",
            remaining_y.count("Yes")
        )


    return rules


# ============================================================
# 7. RUN SEQUENTIAL COVERING
# ============================================================

rules = sequential_covering(
    X,
    y,
    attributes
)


# ============================================================
# 8. DISPLAY FINAL RULES
# ============================================================

print("\n\n")
print("=" * 80)
print("FINAL RULE SET")
print("=" * 80)


for i, rule in enumerate(rules, start=1):

    conditions = []

    for attribute, value in zip(
        attributes,
        rule
    ):

        if value != "?":

            conditions.append(
                f"{attribute} = {value}"
            )


    if len(conditions) == 0:

        condition_string = "TRUE"

    else:

        condition_string = " AND ".join(
            conditions
        )


    print(
        f"\nRule {i}:"
    )

    print(
        f"IF {condition_string}"
    )

    print(
        "THEN Class = Yes"
    )


# ============================================================
# 9. FUNCTION TO PREDICT USING THE LEARNED RULES
# ============================================================

def predict(instance, rules):

    for rule in rules:

        if covers(rule, instance):
            return "Yes"

    return "No"


# ============================================================
# 10. TEST THE LEARNED RULES
# ============================================================

print("\n\n")
print("=" * 80)
print("PREDICTIONS ON TRAINING DATA")
print("=" * 80)


correct = 0


for instance, actual in zip(X, y):

    prediction = predict(
        instance,
        rules
    )

    print("\nInstance :", instance)
    print("Actual   :", actual)
    print("Predicted:", prediction)


    if prediction == actual:
        correct += 1


accuracy_value = (
    correct / len(X)
) * 100


print("\n")
print("=" * 80)
print(f"TRAINING ACCURACY: {accuracy_value:.2f}%")
print("=" * 80)

DATASET
   Sky AirTemp Humidity   Wind Water Forecast Class
 Sunny    Warm   Normal Strong  Warm     Same   Yes
 Sunny    Warm     High Strong  Warm     Same   Yes
 Rainy    Cold     High Strong  Warm   Change    No
 Sunny    Warm     High Strong  Cool   Change   Yes
Cloudy    Warm   Normal   Weak  Warm     Same   Yes
 Rainy    Cold   Normal   Weak  Cool   Change    No
 Sunny    Cold   Normal   Weak  Warm     Same    No
Cloudy    Warm     High Strong  Cool   Change   Yes



################################################################################
SEQUENTIAL COVERING
################################################################################


LEARN-ONE-RULE

Initial rule:
('?', '?', '?', '?', '?', '?')

Current best rule:
('?', 'Warm', '?', '?', '?', '?')
Positive examples covered: 5
Negative examples covered: 0

Learned Rule:
('?', 'Warm', '?', '?', '?', '?')
Positive examples covered: 5
Remaining positive examples: 0



FINAL RULE SET

Rule 1:
IF AirTemp = Warm
THEN Class